In [103]:
import torch
import torch.nn.functional as F
from torch.utils.data import random_split, DataLoader
import matplotlib.pyplot as plt
import numpy as np
%matplotlib inline

In [3]:
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [4]:
len(words)

32033

In [5]:
# build stoi and itos dicts
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [ ]:
# building the dataset
block_size = 3
xs, ys = [], []

for word in words:
    word = '.' * block_size + word + '.'
    for i in range(len(word) - block_size):
        xs.append([stoi[c] for c in list(word[i:i+block_size])])
        ys.append(stoi[word[i+block_size]])

X = torch.tensor(xs)
Y = torch.tensor(ys)

In [ ]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([228146, 3]), torch.int64, torch.Size([228146]), torch.int64)

In [11]:
X

tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 13],
        ...,
        [26, 26, 25],
        [26, 25, 26],
        [25, 26, 24]])

In [ ]:
# creating the lookup/embedding table into 2 dimensions
C = torch.randn((27, 2))

In [9]:
C

tensor([[ 0.2077, -2.3110],
        [ 0.8561,  0.3547],
        [-1.3531,  1.1004],
        [ 0.7678, -0.1514],
        [ 0.1834, -0.8096],
        [ 0.3989, -1.1124],
        [-0.8572, -0.6497],
        [ 0.2540, -1.0081],
        [ 1.4472,  0.0540],
        [ 0.9530, -0.1650],
        [-1.2239, -0.1911],
        [-0.6931, -0.0870],
        [ 0.8472, -0.1697],
        [ 1.6142,  1.1149],
        [-0.8464, -0.1934],
        [-0.1796, -1.4180],
        [-0.2863,  0.4425],
        [-1.0232,  0.5802],
        [-0.0181,  0.8343],
        [-0.9219,  1.6710],
        [-0.5377, -0.1865],
        [-2.0903,  0.4545],
        [-0.4511, -1.3059],
        [ 0.4554,  0.0775],
        [ 1.0842, -1.4115],
        [-1.2940, -0.8218],
        [-1.5206, -1.8012]])

In [ ]:
# embedding the data by indexing by tensor X (replace all indices in X with their corresponding element in C)
emb = C[X]
print(emb.shape)

torch.Size([228146, 3, 2])


In [14]:
# creating the weights and biases of the 100 neurons in the hidden layer
W1 = torch.randn((6,100))
b1 = torch.randn(100)

In [42]:
# the issue is we want a single vector as input to the hidden layer but we have 3x2 matrices
# .view() is a very fast way of casting to desired dimensions.
# calculate outputs of the hidden layer
h = torch.tanh(emb.view(len(X),6) @ W1 + b1)


In [43]:
h

tensor([[-1.0000, -1.0000,  0.0657,  ..., -1.0000, -0.9884,  1.0000],
        [-1.0000, -0.9999,  0.6947,  ..., -1.0000, -0.9881,  1.0000],
        [-1.0000, -0.9745,  0.9998,  ..., -0.9999, -0.9745,  1.0000],
        ...,
        [-0.9835, -0.9979, -0.1158,  ..., -1.0000, -0.6017,  1.0000],
        [-0.9882, -1.0000, -0.2603,  ..., -1.0000, -0.6028,  1.0000],
        [-0.9979, -0.9942,  0.9913,  ..., -1.0000,  0.8448,  1.0000]])

In [44]:
# creating weights and biases of the output layer
W2 = torch.randn((100,27))
b2 = torch.randn(27)


In [45]:
# calculate outputs of the output layer
logits = h @ W2 + b2

In [46]:
logits

tensor([[ 17.7058,  -7.4539,   6.6085,  ...,  13.3743,   3.6811,   4.6977],
        [ 18.3623,  -6.4212,   5.9916,  ...,  13.7899,   4.6964,   7.5800],
        [  5.8517,  -4.5493,   3.0896,  ...,   6.3954,   1.2220,   3.6790],
        ...,
        [ 14.6565,  -7.8044,   1.0642,  ...,  18.2203,   6.2493,   8.0132],
        [  5.7692, -12.3006,   2.6504,  ...,  14.7506,   8.0887,   9.1869],
        [ 11.5710,   3.3850,   2.1247,  ...,   6.0757,   6.9182,   9.2325]])

In [50]:
# calculating NLL
counts = logits.exp()
probs = counts / counts.sum(1, keepdim=True)
sum(probs[2])


tensor(1.)

In [51]:
# calculating loss using F.cross_entropy
loss = F.cross_entropy(logits, Y)
loss

tensor(19.7425)

In [54]:
# ---------- NOW MADE RESPECTABLE ----------

In [299]:
# hyperparameters
block_size = 4
emb_dim = 12
num_h_neurons = 200
initial_learning_rate = 0.1
decayed_learning_rate = 0.01
minibatch_size = 100
initial_training_iters = 200000
decayed_training_iters = 10000
total_training_iters = initial_training_iters + decayed_training_iters

In [300]:
# building the dataset
xs, ys = [], []

for word in words:
    word = '.' * block_size + word + '.'
    for i in range(len(word) - block_size):
        xs.append([stoi[c] for c in list(word[i:i+block_size])])
        ys.append(stoi[word[i+block_size]])

X = torch.tensor(xs)
Y = torch.tensor(ys)

In [301]:
# prettifying / setting up variables
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27,emb_dim), generator=g)
W1 = torch.randn((block_size * emb_dim, num_h_neurons), generator=g)
b1 = torch.randn(num_h_neurons, generator=g)
W2 = torch.randn((num_h_neurons,27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [302]:
sum(p.nelement() for p in parameters) # total num elems

15551

In [303]:
# creating train, dev, test split
n = X.shape[0]
perm = torch.randperm(n, generator=g)

X_shuffled = X[perm]
Y_shuffled = Y[perm]

n1 = int(0.8 * n)
n2 = int(0.9 * n)

X_train, Y_train = X_shuffled[:n1], Y_shuffled[:n1]
X_dev, Y_dev = X_shuffled[n1:n2], Y_shuffled[n1:n2]
X_test, Y_test = X_shuffled[n2:], Y_shuffled[n2:]

In [304]:
for p in parameters:
    p.requires_grad = True

# training loop
for i in range(total_training_iters):

    # create minibatch
    ix = torch.randint(0, X_train.shape[0], (minibatch_size,), generator=g)

    # forward pass
    emb = C[X_train[ix]]
    h = torch.tanh(emb.view(-1, block_size * emb_dim) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Y_train[ix])

    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    # update params
    for p in parameters:
        assert p.grad is not None
        p.data -= (initial_learning_rate if i < initial_training_iters else decayed_learning_rate) * p.grad

# evaluating train_loss
emb = C[X_train]
h = torch.tanh(emb.view(-1, block_size * emb_dim) @ W1 + b1)
logits = h @ W2 + b2
train_loss = F.cross_entropy(logits, Y_train)
print("Train loss:", train_loss.item())

Train loss: 2.056487560272217


In [306]:
# evaluating dev_loss
emb = C[X_dev]
h = torch.tanh(emb.view(-1, block_size * emb_dim) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Y_dev)
print("Dev loss:", loss.item())

Dev loss: 2.1325809955596924


In [ ]:
# ---------- Results ----------

Hyperparameters:

block_size = 3
emb_dim = 10
num_h_neurons = 200
initial_learning_rate = 0.1
decayed_learning_rate = 0.01
minibatch_size = 500
initial_training_iters = 100000
decayed_training_iters = 10000
total_training_iters = initial_training_iters + decayed_training_iters

Train loss: 2.128025770187378

Dev loss: 2.1724913120269775

Hyperparameters:
block_size = 4
emb_dim = 12
num_h_neurons = 200
initial_learning_rate = 0.1
decayed_learning_rate = 0.01
minibatch_size = 100
initial_training_iters = 200000
decayed_training_iters = 10000
total_training_iters = initial_training_iters + decayed_training_iters

Train loss: 2.056487560272217

Dev loss: 2.1325809955596924